In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv('./dataset/original_data.csv')
df.head()

,id,title,abstract,full_text
0,1909.00694,Minimally Supervised Learning of Affective Eve...,Recognizing affective events that trigger posi...,"{'section_name': ['Introduction', 'Related Wor..."
1,2003.07723,"PO-EMO: Conceptualization, Annotation, and Mod...",Most approaches to emotion analysis regarding ...,"{'section_name': ['', ' ::: ', ' ::: ::: ', '..."
2,1705.09665,Community Identity and User Engagement in a Mu...,A community's identity defines and shapes its ...,"{'section_name': ['Introduction', 'A typology ..."
3,1908.06606,Question Answering based Clinical Text Structu...,Clinical text structuring is a critical and fu...,"{'section_name': ['Introduction', 'Related Wor..."
4,1811.00942,Progress and Tradeoffs in Neural Language Models,"In recent years, we have witnessed a dramatic ...","{'section_name': ['Introduction', 'Background ..."


# 数据清洗

In [2]:
import ast

# 取第一行数据解析
raw_text = df.iloc[0]['full_text']
data = ast.literal_eval(raw_text) # 还原成对象

print("【数据结构解剖报告】")
print(f"1. 最外层类型是: {type(data)}") # 应该是 <class 'dict'>

if isinstance(data, dict):
    print(f"2. 字典里所有的 Keys: {list(data.keys())}")
    
    # 看看 section_name 对应的是什么
    sec_val = data.get('section_name')
    print(f"3. section_name 对应值的类型: {type(sec_val)}") # 应该是 <class 'list'>
    print(f"   预览前2个章节名: {sec_val[:2]}")
    
    # 看看 paragraphs 对应的是什么
    para_val = data.get('paragraphs')
    if para_val:
        print(f"4. paragraphs 对应值的类型: {type(para_val)}")   # 应该是 <class 'list'>
        print(f"   列表长度: {len(para_val)}")
        print(f"   第一个段落的内容类型: {type(para_val[0])}") # 里面可能还套了一层 list
else:
    print("数据不是字典，推断错误。")


【数据结构解剖报告】
1. 最外层类型是: <class 'dict'>
2. 字典里所有的 Keys: ['section_name', 'paragraphs']
3. section_name 对应值的类型: <class 'list'>
   预览前2个章节名: ['Introduction', 'Related Work']
4. paragraphs 对应值的类型: <class 'list'>
   列表长度: 21
   第一个段落的内容类型: <class 'list'>


In [4]:
all_chunks = []

print("开始处理数据...")

for index, row in df.iterrows():
    try:
        if isinstance(row['full_text'], str):
            full_text_data = ast.literal_eval(row['full_text'])
        else:
            full_text_data = row['full_text']
        
        # 根据预览，数据结构是一个大字典，包含两个列表
        # 结构为: {'section_name': ['文本1', '文本2'], 'paragraphs': [['文本1'], ['文本2']]}
        if isinstance(full_text_data, dict):
            # 获取所有段落名列表
            sec_names = full_text_data.get('section_name', [])
            # 获取所有段落内容列表
            sec_paras = full_text_data.get('paragraphs', [])
            
            # 使用 zip 同时遍历两个列表
            if len(sec_names) == len(sec_paras):
                for name, para_list in zip(sec_names, sec_paras):
                    # para_list 也是一个列表，需要拼接
                    text_content = " ".join(para_list) if isinstance(para_list, list) else str(para_list)
                    
                    all_chunks.append({
                        "paper_id": row['id'],
                        "title": row['title'],
                        "section_name": name,
                        "text": text_content
                    })
            else:
                print(f"警告: ID {row['id']} 的标题和内容数量不匹配，跳过。")

    except Exception as e:
        print(f"Error at index {index}: {e}")
        continue

print(f"处理完成！共获得 {len(all_chunks)} 个段落章节。")

# 检查一下前3个结果
if len(all_chunks) > 0:
    print("-" * 30)
    print("【结果预览 (前3条)】:")
    for i in range(min(3, len(all_chunks))):
        print(f"[{i}] ID: {all_chunks[i]['paper_id']} | Section: {all_chunks[i]['section_name']}")

开始处理数据...
处理完成！共获得 21909 个段落章节。
------------------------------
【结果预览 (前3条)】:
[0] ID: 1909.00694 | Section: Introduction
[1] ID: 1909.00694 | Section: Related Work
[2] ID: 1909.00694 | Section: Proposed Method


# 二次切分与文本构造

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm

# 1. 定义切分器
# chunk_size=1000: 大约对应 700-800 个单词，是一个比较安全的长度
# chunk_overlap=100: 保持上下文连贯
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

processed_docs = []

print("正在进行二次切分与格式化...")

for chunk in tqdm(all_chunks):
    # 构造包含丰富语义的文本
    # 格式：Title: [标题] \n Section: [章节名] \n Content: [内容]
    base_text = f"Title: {chunk['title']}\nSection: {chunk['section_name']}\nContent: {chunk['text']}"
    
    # 检查长度
    if len(base_text) > 1000:
        # 如果太长，就切分
        split_texts = text_splitter.split_text(base_text)
        for i, sub_text in enumerate(split_texts):
            processed_docs.append({
                "index_id": len(processed_docs), # 全局唯一索引ID
                "paper_id": chunk['paper_id'],
                "section_name": chunk['section_name'],
                "text_content": sub_text, # 切分后的文本
                "metadata": chunk # 保留原始元数据备用
            })
    else:
        # 如果长度合适，直接用
        processed_docs.append({
            "index_id": len(processed_docs),
            "paper_id": chunk['paper_id'],
            "section_name": chunk['section_name'],
            "text_content": base_text,
            "metadata": chunk
        })

print(f"切分完成！最终索引库大小: {len(processed_docs)} (原段落数: {len(all_chunks)})")

正在进行二次切分与格式化...


100%|████████████████████████████████████████| 21909/21909 [00:10<00:00, 2030.64it/s]

切分完成！最终索引库大小: 64527 (原段落数: 21909)


# 向量化

In [9]:
from sentence_transformers import SentenceTransformer
import torch

# 检查设备
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"正在使用设备: {device}")

# 1. 加载模型
# 'BAAI/bge-m3' 是目前最强的开源检索模型之一，支持多语言和长文本
# 如果下载太慢或显存不够，可以换成 'BAAI/bge-small-en-v1.5'
model_name = 'BAAI/bge-m3' 
print(f"正在加载模型: {model_name} ...")
model = SentenceTransformer(model_name, device=device)

# 2. 准备文本列表
sentences = [doc['text_content'] for doc in processed_docs]

# 3. 开始转换 (Encoding)
print("开始生成向量 (这可能需要几分钟)...")
# batch_size: 根据你的显存调整，8G显存通常设 16 或 32
embeddings = model.encode(sentences, batch_size=16, show_progress_bar=True, normalize_embeddings=True)

print(f"向量生成完毕！形状: {embeddings.shape}")
# 形状应该是 (文档数, 1024)

正在使用设备: cuda
正在加载模型: BAAI/bge-m3 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

F:\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Administrator\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

开始生成向量 (这可能需要几分钟)...


Batches:   0%|          | 0/4033 [00:00<?, ?it/s]

向量生成完毕！形状: (64527, 1024)


# 构建索引并保存

In [ ]:
import faiss
import pickle

# 1. 构建 FAISS 索引
d = embeddings.shape[1] # 向量维度 (例如 1024)
# 使用最简单的内积索引 (Inner Product)，因为我们已经 normalize 过了，所以等价于余弦相似度
index = faiss.IndexFlatIP(d) 

print("正在构建索引...")
index.add(embeddings) # 将向量添加进索引
print(f"索引构建完成，当前索引内包含 {index.ntotal} 条数据。")

# 2. 保存结果到硬盘 (持久化)
# 保存索引
faiss.write_index(index, "qasper_faiss.index")
# 保存文本数据 (因为索引里只存了向量，没存字，我们需要把字也存起来对应)
with open("qasper_docs.pkl", "wb") as f:
    pickle.dump(processed_docs, f)

print("✅ 所有数据已保存到本地：")
print("   1. qasper_faiss.index (向量索引)")
print("   2. qasper_docs.pkl (文本数据库)")